# Configuration

In [ ]:
import os
#os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)
import torch
from datasets import load_dataset, ClassLabel, Value, Dataset
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    PeftModel
)
from trl import DPOTrainer, DPOConfig

# Set Global Variables

In [ ]:
#Model and Tokenizer
model_id = "PATH/TO/LOCAL/MODEL"
peft_path = "PATH/TO/SFT/MODEL"
model_name = "MODEL NAME"

#QLORA
r = 8
lora_alpha = 8

#Training
learning_rate = 1e-07
beta = 0.7
output_dir = f'PATH/TO/OUTPUT/DIR'

# Data & Formatting

In [ ]:
training_data = pd.read_csv("PATH/TO/TRAINING/DATA") #Data file should be formatted with 3 columns: "prompt", "chosen", "rejected" all of which are strings
dataset = Dataset.from_pandas(training_data)
dataset = dataset.train_test_split(test_size = 0.2)

# Functions

In [ ]:
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.reward_margin = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if "loss" in logs:
            self.train_losses.append(logs["loss"])
        if "eval_loss" in logs:
            self.eval_losses.append(logs["eval_loss"])
        if "eval_rewards/margins" in logs:
            self.reward_margin.append(logs["eval_rewards/margins"])

# Model & Tokenizer Loading

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant = True
)

model = AutoModelForCausalLM.from_pretrained(
        model_id,
        attn_implementation = "flash_attention_2",
        low_cpu_mem_usage=True,
        torch_dtype=torch.bfloat16,
        quantization_config = bnb_config,
        device_map = 'auto'
    )

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# When Training smaller model this method works but for larger takes up too much space

'''
ref_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        low_cpu_mem_usage=True,
        torch_dtype=torch.bfloat16,
        quantization_config = bnb_config,
        device_map = 'auto'
    ).eval()
'''

# QLORA

In [ ]:
model = PeftModel.from_pretrained(model, peft_path)
model = model.merge_and_unload()

In [ ]:
peft_config = LoraConfig(
        lora_alpha = lora_alpha,
        lora_dropout = 0.1,
        r = r,
        bias = "none",
        task_type = "CAUSAL_LM",
        target_modules =  ['q_proj', 'v_proj', 'k_proj', 'o_proj']
    )

model = get_peft_model(model, peft_config)
#model.load_adapter(model_id, adapter_name = "reference")

# Training

In [ ]:
loss_logger = LossLoggerCallback()

training_args = DPOConfig(
        #Base Training Stuff
        num_train_epochs = 20, #Doesn't take this many, but trains so slow that it was usually manually stopped
        beta = beta,
        learning_rate = learning_rate,
        warmup_ratio = 0.1,
        optim = "adamw_torch",
        #Batch & Gradient
        per_device_train_batch_size = 1, #Change as GPU resources allow
        per_device_eval_batch_size = 1, #Change as GPU resources allow
        gradient_accumulation_steps = 1, #Change as GPU resources allow
        gradient_checkpointing = True,
        #Saving
        save_strategy= "steps",
        output_dir = output_dir,
        save_steps = 10,
        #Logging
        logging_strategy = "steps",
        logging_dir = f"{output_dir}/logs",
        logging_steps = 10,
        #Eval
        do_eval = True,
        eval_strategy= "steps",
        eval_steps = 10,
        #Callback
        load_best_model_at_end = True,
        metric_for_best_model = "eval_rewards/margins",
        greater_is_better = False,
        #DPO Specific
        max_length = 13768, #Change if needed
        max_prompt_length = 12768, #Change if needed
        #model_adapter_name = "default",
        #ref_adapter_name = "reference"
)

dpo_trainer = DPOTrainer(
        model,
        ref_model = None,
        args = training_args,
        train_dataset = dataset['train'],
        eval_dataset = dataset['test'],
        tokenizer = tokenizer,
        callbacks = [loss_logger]

    )

In [ ]:
#Train
print("Training Model...")
torch.cuda.empty_cache()
dpo_trainer.train()

#Save Stuff
output_dir = os.path.join(output_dir, "final_checkpoint")
print("Saving Pretrained Model...")
dpo_trainer.model.save_pretrained(output_dir)
print("Saving Pretrained Tokenizer...")
tokenizer.save_pretrained(output_dir)